### Part 2 - Question 1

In [1]:
import numpy as np

# (a) Store the input and filter as NumPy arrays
image = np.array([
    [1, 1, 1, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 1, 1, 1],
    [0, 0, 1, 1, 0],
    [0, 1, 1, 0, 0],
])
kernel = np.array([
    [1, 0, 1],
    [0, 1, 0],
    [1, 0, 1],
])

def conv2d(x, k, stride=1, padding=0):
    """Slide kernel k over x and take the dot product at every valid location."""
    if padding > 0:
        x = np.pad(x, padding, mode="constant")      # zero padding
    n, f = x.shape[0], k.shape[0]
    out_size = (n - f) // stride + 1                  # Nout = (N - F + 2P)/S + 1 (P already applied)
    out = np.zeros((out_size, out_size), dtype=int)

    # (b) slide the filter over the input
    for i in range(out_size):
        for j in range(out_size):
            r, c = i * stride, j * stride
            window = x[r:r + f, c:c + f]              # current receptive field
            out[i, j] = np.sum(window * k)            # (c) dot product (element-wise multiply, then sum)
    return out


if __name__ == "__main__":
    # Stride = 1, padding = 0
    result = conv2d(image, kernel, stride=1, padding=0)
    print("Output feature map (stride=1, padding=0):")   # (d)
    print(result)
    print("Output shape:", result.shape)                 # (e)

    # Extra: stride = 2 for part (f)
    result_s2 = conv2d(image, kernel, stride=2, padding=0)
    print("\nFor comparison, stride=2:")
    print(result_s2)
    print("Output shape:", result_s2.shape)

    print("""
(f) Explanation: With stride 2 the filter jumps two pixels at a time instead of one,
so it visits fewer positions. Nout = (5 - 3)/2 + 1 = 2, so the output shrinks from
3x3 to 2x2. Stride > 1 downsamples the feature map (less computation, fewer values,
but some spatial detail is skipped).""")

Output feature map (stride=1, padding=0):
[[4 3 4]
 [2 4 3]
 [2 3 4]]
Output shape: (3, 3)

For comparison, stride=2:
[[4 4]
 [2 4]]
Output shape: (2, 2)

(f) Explanation: With stride 2 the filter jumps two pixels at a time instead of one,
so it visits fewer positions. Nout = (5 - 3)/2 + 1 = 2, so the output shrinks from
3x3 to 2x2. Stride > 1 downsamples the feature map (less computation, fewer values,
but some spatial detail is skipped).


### Part 2 - Question 2

In [2]:
%pip install torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 MB 56.7 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 51.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 62.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 25.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [torchvision] [torchvision]
Note: you may need to restart the kernel to use updated packages.


In [14]:
import json
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

# ---------------- Configuration ----------------
SEED = 42
EPOCHS = 5                  # SAME number of epochs for both experiments
BATCH_SIZE = 64
LR = 1e-3
TRAIN_PER_CLASS = 1000      # subset size per class (keeps CPU training fast)
TEST_PER_CLASS = 250
IMG_SIZE = 128              # upscale 32x32 CIFAR images for the pretrained network
CLASSES = {3: 0, 5: 1}      # CIFAR-10 label 3 = cat, 5 = dog -> new labels 0/1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)

# ---------------- Data ----------------
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet statistics
tfm = transforms.Compose([transforms.Resize(IMG_SIZE), transforms.ToTensor(), norm])


class TwoClass:
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        x, y = self.dataset[self.indices[i]]

        # Convert CIFAR-10 labels
        # cat = 3 -> 0
        # dog = 5 -> 1
        y = CLASSES[y]

        return x, y


train_base = datasets.CIFAR10("./data", train=True, download=True, transform=tfm)
test_base = datasets.CIFAR10("./data", train=False, download=True, transform=tfm)
train_loader = DataLoader(TwoClass(train_base, pick(train_base, TRAIN_PER_CLASS)),
                          batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TwoClass(test_base, pick(test_base, TEST_PER_CLASS)),
                         batch_size=BATCH_SIZE)


# ---------------- Helpers ----------------
def build_model(fine_tune):
    """Load pretrained ResNet18 and set which layers are trainable."""
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for p in model.parameters():                 # freeze everything first
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, 2)  # new classifier (trainable by default)
    if fine_tune:                                # Exp. B: also unfreeze the last conv block
        for p in model.layer4.parameters():
            p.requires_grad = True
    return model.to(DEVICE)


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def evaluate(model):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total


def run_experiment(name, fine_tune):
    model = build_model(fine_tune)
    trainable = count_trainable(model)
    print(f"\n=== {name} === trainable parameters: {trainable:,}")
    # Only pass trainable parameters to the optimizer
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        running, n = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()
            running += loss.item() * y.size(0); n += y.size(0)
        losses.append(running / n)
        print(f"epoch {epoch + 1}/{EPOCHS}  train loss {losses[-1]:.4f}")
    train_time = time.time() - start
    acc = evaluate(model)
    print(f"training time {train_time:.1f}s   test accuracy {acc:.2f}%")
    return {"name": name, "trainable": trainable, "time": train_time, "acc": acc, "losses": losses}


# ---------------- Run both experiments ----------------
if __name__ == "__main__":
    a = run_experiment("Frozen Feature Extractor", fine_tune=False)
    b = run_experiment("Fine-Tuned Network", fine_tune=True)

    # (e) Plot training loss
    plt.figure(figsize=(6, 4))
    for r in (a, b):
        plt.plot(range(1, EPOCHS + 1), r["losses"], marker="o", label=r["name"])
    plt.xlabel("Epoch"); plt.ylabel("Training loss"); plt.title("Training loss: frozen vs fine-tuned")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig("training_loss.png", dpi=150)

    # (f) Comparison table
    table = ("| Method | Trainable Parameters | Training Time | Accuracy |\n|---|---|---|---|\n"
             + "".join(f"| {r['name']} | {r['trainable']:,} | {r['time']:.1f} s | {r['acc']:.2f}% |\n"
                       for r in (a, b)))
    print("\n" + table)
    open("results_table.md", "w").write(table)
    json.dump([a, b], open("results.json", "w"), indent=2)


=== Frozen Feature Extractor === trainable parameters: 1,026
epoch 1/5  train loss 0.6266
epoch 2/5  train loss 0.4398
epoch 3/5  train loss 0.3887
epoch 4/5  train loss 0.3798
epoch 5/5  train loss 0.3556
training time 95.6s   test accuracy 80.60%

=== Fine-Tuned Network === trainable parameters: 8,394,754
epoch 1/5  train loss 0.5298
epoch 2/5  train loss 0.1333
epoch 3/5  train loss 0.0777
epoch 4/5  train loss 0.0210
epoch 5/5  train loss 0.0087
training time 125.1s   test accuracy 86.00%

| Method | Trainable Parameters | Training Time | Accuracy |
|---|---|---|---|
| Frozen Feature Extractor | 1,026 | 95.6 s | 80.60% |
| Fine-Tuned Network | 8,394,754 | 125.1 s | 86.00% |

